In [19]:
#----import necessary documents-----
import hashlib
from pathlib import Path

#----Langchain tools for Extract and Transform pipeline-----
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#----Qdrant libraries for the Load phase-----
from qdrant_client import QdrantClient
from qdrant_client.http import models

#---import necessary files from src-----

from config import settings
from schemas import ChunkMetadata


In [14]:
path_page = Path("/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/data/Operating System -Chap 1 - Eng.pdf")
pages = PyPDFLoader(path_page).load()
print(pages[10].page_content)
print("-"*10)
print(pages[10].metadata)

• One/ many CPUs, controlling devices are linked by a common bus system to 
access a shared memory.
• Controlling devices and CPU operate simultaneously and compete with each 
other.
Input Output 
System Memory
Bus
Processor
A computer system’s structure
Chapter 1. Operating System Overview
1. Operating system concept
             1.1. Layered structure of a computing system
----------
{'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-09-30T10:11:26+07:00', 'title': 'Hệ Điều Hành (Nguyên lý các hệ điều hành)', 'author': 'Quoc Huy', 'moddate': '2025-09-30T10:11:26+07:00', 'source': '/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/data/Operating System -Chap 1 - Eng.pdf', 'total_pages': 179, 'page': 10, 'page_label': '11'}


In [ ]:
def _document_id(path):
    raw = f"{path.name}:{path.stat().st_size}"
    hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

print(_document_id(path_page))

4e4943ce07360667


In [ ]:
def _chunk_id(doc_id, page, index) -> str:
    return f"{doc_id}:{page}:{index}"

def _load_pdf(path):

    pages = PyPDFLoader(str(path)).load()
    doc_id = _document_id(path)

    for doc in pages:
        page_number = doc.metadata.get("page", 0) + 1 # +1 to make page number human-readable (1-indexed)
        doc.metadata = {
            "document_id": doc_id, 
            "filename": path.name,
            "source": str(path.resolve()),
            "page": page_number,
            "section": doc.metadata.get("section")
        }

    return pages

def _splitter(chunk_size = None, chunk_overlap = None):
    """
    build the recursive splitter function
    """
    return RecursiveCharacterTextSplitter(
        chunk_size = chunk_size or settings.chunk_size,
        chunk_overlap = chunk_overlap or settings.chunk_overlap,
        separators = ["\n\n", "\n", ".", " ", ""],
        keep_separator = False
    )


    





[Document(metadata={'document_id': '4e4943ce07360667', 'filename': 'Operating System -Chap 1 - Eng.pdf', 'source': '/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/data/Operating System -Chap 1 - Eng.pdf', 'page': 1, 'section': None}, page_content='Operating Systems\n(Principles of Operating Systems)\nĐỗ Quốc Huy\nhuydq@soict.hust.edu.vn\nDepartment of Computer Science \nSchool of Information and Communication Technology'), Document(metadata={'document_id': '4e4943ce07360667', 'filename': 'Operating System -Chap 1 - Eng.pdf', 'source': '/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/data/Operating System -Chap 1 - Eng.pdf', 'page': 2, 'section': None}, page_content='Course Info\nPrerequisite:\n• Introduction to information technology, Data structure \nand algorithm\n• C programming skill'), Document(metadata={'document_id': '4e4943ce07360667', 'filename': 'Operating System -Chap 1 - Eng.pdf', 'source': '/Users/doanhaidang1703/Documents/A

In [1]:
'''
This file is used to prepare for the loading, chunking, and saving into vector database pipeline
First we use the PyPDFLoader --> Recursive splitter --> Embedding --> Saving/ Qdrant - Vector Store
'''
import uuid
from store import get_vector_store, ensure_collection
import hashlib
from pathlib import Path
from collections import defaultdict

#---- import libraries for extract and transform pipeline of ETL from Langchain -----
# pyrefly: ignore [missing-import]
from langchain_community.document_loaders import PyPDFLoader
# pyrefly: ignore [missing-import]
from langchain_text_splitters import RecursiveCharacterTextSplitter

#--- import modules from our projects -----
from config import settings
from schemas import ChunkMetadata


def discover_pdfs():
    '''
    search through data folder, return lists of paths of pdfs that will be used to build the pipeline.
    '''
    directory_path = settings.data_dir
    pdf_paths = list(directory_path.glob("*.pdf"))
    
    return pdf_paths




#---- build functions to handle loading pdf files and assign metadata with document_id, filename, source, page and section ----
def _document_id(path) -> str:
    """Generate a document ID from a file path and its size in byte"""
    raw = f"{path.name}{path.stat().st_size}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

def _chunk_id(doc_id, page, index) -> str:
    return f"{doc_id}:{page}:{index}"

def _load_pdf(path):

    pages = PyPDFLoader(str(path)).load()
    doc_id = _document_id(path)

    for doc in pages:
        page_number = doc.metadata.get("page", 0) + 1 # +1 to make page number human-readable (1-indexed)
        doc.metadata = {
            "document_id": doc_id, 
            "filename": path.name,
            "source": str(path.resolve()),
            "page": page_number,
            "section": doc.metadata.get("section")
        }

    return pages

def _splitter(chunk_size = None, chunk_overlap = None):
    """
    build the recursive splitter function
    """
    return RecursiveCharacterTextSplitter(
        chunk_size = chunk_size or settings.chunk_size,
        chunk_overlap = chunk_overlap or settings.chunk_overlap,
        separators = ["\n\n", "\n", ".", " ", ""],
        keep_separator = False
    )


def build_chunks(pdf_paths, chunk_size = None, chunk_overlap = None, chunker = None):

    page_docs = []
    for path in pdf_paths:
        page_docs.extend(_load_pdf(path))
    
    splitter = chunker or _splitter(chunk_size, chunk_overlap)
    chunks = splitter.split_documents(page_docs)

    #initialize a defaultdict to keep track of the number of chunks for each document.
    #when it sees a new document_id, it automatically initializes the count to 0
    #if page 1 has 3 chunks, when adding new chunks on different pages, it should start counting from the next number after the last count on that specific document.
    per_doc_counter = defaultdict(int)

    for chunk in chunks:
        doc_id = chunk.metadata["document_id"]
        idx = per_doc_counter[doc_id]
        per_doc_counter[doc_id] += 1

        meta = ChunkMetadata(
            document_id = doc_id,
            filename = chunk.metadata["filename"],
            source = chunk.metadata["source"],
            page = chunk.metadata["page"],
            chunk_id = _chunk_id(doc_id, chunk.metadata["page"], idx),
            section = chunk.metadata.get("section")
        )
        
        chunk.metadata = meta.model_dump()

    return chunks

def index_chunks(chunks, collection_name):
    '''
    this function add chunks into database after checking if there is any chunk in the database
    Args:
        chunks: 
        collection_name: 
    
    Returns:
        len(chunks)
    '''

    if not chunks:
        return 0
    #create chunk id to uniquely identify each chunk in the database
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"])) for c in chunks]
    get_vector_store(collection_name = collection_name).add_documents(documents = chunks, ids = ids)
    return len(chunks)
    
def ingest(recreate = False, collection_name = None, chunk_size = None, chunker = None, chunk_overlap = None):
    #ensure the collection and index exists
    pdf_paths = discover_pdfs()
    ensure_collection(
        recreate = recreate,
        collection_name = collection_name
    )
    
    chunks = build_chunks(
        pdf_paths = pdf_paths,
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        chunker = chunker
    )

    return index_chunks(chunks, collection_name)

def save_and_ingest(file_bytes, filename):
    safe_name = Path(filename).name
    dest = settings.data_dir / safe_name
    settings.data_dir.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(file_bytes)
    ensure_collection(recreate=False)
    chunks = build_chunks([dest])
    return {"filename": safe_name, "chunks_indexed": index_chunks(chunks)}
    

print(ingest())


/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/.rag_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/f3/jwhj4vkx0ns8x0_z6d_yb3n80000gn/T/ipykernel_66008/2360439935.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9511.78it/s]
/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/src/store.py:87: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(name, 

178


In [3]:
from config import settings
print(settings.data_dir.resolve())

/Users/doanhaidang1703/Documents/AIO/AIO Code/Module 12/rag/notebook_lm/data
